# mechanism_viewer examples

## Generate missing data
This notebook shows how to use functions from `dataset_generator.py` to generate complete data with different feature types, and then transform the dataset so some or all features follow missing-data mechanisms (MCAR/MAR/MNAR).

For reproducible experiments, the examples explicitly use `random_state=DEFAULT_RANDOM_STATE` when applicable. The functions that use this parameter are `generate_synthetic_dataset()`, `apply_mcar()`, `apply_missing_data()`, and `generate_dataset_with_missing_data()`.

### 1. Generate complete data with `generate_synthetic_dataset()`

Use `generate_synthetic_dataset()` to create a complete dataset with a specific number of rows. The number of columns and each feature type are chosen with an array of supported data-type strings.

In [ ]:
import pandas as pd
from mechanism_viewer import generate_synthetic_dataset, ColType, DEFAULT_RANDOM_STATE

n_rows = 100
complete_data = generate_synthetic_dataset(n_rows, [ColType.CONTINUOUS, ColType.CONTINUOUS, ColType.DISCRETE, ColType.DISC_CATEGORICAL, ColType.BINARY],  random_state=DEFAULT_RANDOM_STATE)
display(complete_data.head(10))

,Col1,Col2,Col3,Col4,Col5
0,0.496714,-1.415371,11,2,1
1,-0.138264,-0.420645,7,0,1
2,0.647689,-0.342715,3,7,0
3,1.523030,-0.802277,2,2,1
4,-0.234153,-0.161286,3,1,1
5,-0.234137,0.404051,4,0,1
6,1.579213,1.886186,5,2,1
7,0.767435,0.174578,5,1,0
8,-0.469474,0.257550,4,1,0
9,0.542560,-0.074446,3,5,1


### 2. Apply missingness on complete data with `apply_missing_data()`

Use `apply_missing_data()` to remove rows from chosen features, according to the desired missing data mechanism and missing rate.

In [ ]:
from mechanism_viewer import apply_missing_data

missing_rate = 0.2
n_complete_cols = 2
data = apply_missing_data(complete_data, n_complete_cols, ["MAR", "MCAR", "MNAR"], [missing_rate, missing_rate, missing_rate])
display(data.head(10))

,Col1,Col2,Col3,Col4,Col5
0,0.496714,-1.415371,11,2,1
1,-0.138264,-0.420645,7,0,1
2,0.647689,-0.342715,3,7,0
3,1.523030,-0.802277,2,<NA>,1
4,-0.234153,-0.161286,3,1,1
5,-0.234137,0.404051,4,<NA>,1
6,1.579213,1.886186,<NA>,2,1
7,0.767435,0.174578,<NA>,1,0
8,-0.469474,0.257550,4,1,0
9,0.542560,-0.074446,3,5,<NA>


Let’s try another dataset. This one has one complete discrete column (`Col1`) and the same missing columns: `Col2` as MAR, `Col3` as MCAR, and `Col4` as MNAR.

In [ ]:
data2 = generate_synthetic_dataset(100, [ColType.DISCRETE ,ColType.DISCRETE, ColType.DISC_CATEGORICAL, ColType.BINARY])

missing_rate = 0.2
n_complete_cols = 1

data2 = apply_missing_data(data2, n_complete_cols, ["MAR", "MCAR", "MNAR"], [missing_rate, missing_rate, missing_rate])

Let’s apply missing data to non-numeric columns. In this example, `animal_col` is converted to numeric values only to define an order, which is then used to choose missing indexes in the original `animal_col` with MNAR.

Meanwhile, `fruit_col` uses the order obtained from converting `color_col` to numeric values, since it is processed with MAR.

Numeric conversion only happens in MAR and MNAR. MCAR uses random sampling to select missing indexes, so it does not require numeric conversion.

In [ ]:
data4 = pd.DataFrame({
    "color_col": ["blue", "yellow", "green", "black", "purple"],
    "numeric_col": [10, 20, 30, 20, 10],
    "fruit_col": ["apple", "banana", "apple", "orange", "banana"],
    "animal_col": ["cat", "dog", "dog", "fish", "cat"]
})

data4 = apply_missing_data(data4, 1, ["MCAR","MAR","MNAR"], [0.6,0.2, 0.4])

display(data4)

,color_col,numeric_col,fruit_col,animal_col
0,blue,NaN,apple,cat
1,yellow,20.0,NaN,NaN
2,green,NaN,apple,dog
3,black,20.0,orange,NaN
4,purple,NaN,banana,cat


### 3. Applying missing data to a single column with `apply_mcar()`, `apply_mar()`, and `apply_mnar()`

To apply a missing-data mechanism to a single column, use `apply_mcar()`, `apply_mar()`, or `apply_mnar()`. Since each mechanism removes rows differently, each function requires different inputs.

In [ ]:
from mechanism_viewer import generate_synthetic_dataset, ColType, apply_mcar, apply_mar, apply_mnar, DEFAULT_RANDOM_STATE

data5 = generate_synthetic_dataset(100, [ColType.DISCRETE, ColType.BINARY, ColType.CONTINUOUS, ColType.CONTINUOUS], random_state=DEFAULT_RANDOM_STATE)

new_Col4 = apply_mcar(data5["Col4"], missing_rate=0.1, random_state=DEFAULT_RANDOM_STATE)

new_Col3 = apply_mar(data5["Col3"], observable_df=data5[["Col1"]], missing_rate=0.1, missingness_ascending=True)

new_Col2 = apply_mnar(data5["Col2"], missing_rate=0.1, missingness_ascending=True)

> Note the argument `missingness_ascending` is optional in missing data mechanisms of MAR and MNAR. This argument changes the order that the missing rows are chosen. It is also available in `apply_missing_data()` and `generate_dataset_with_missing_data()`.

### 4. Generate missing data with `generate_dataset_with_missing_data()`

The function `generate_dataset_with_missing_data()` wraps around `generate_synthetic_dataset()` and `apply_missing_data()` to return a synthetic dataset with missing values.

This example has one complete continuous column (`Col1`) and the same missing columns (`Col2` as MAR, `Col3` as MCAR, and `Col4` as MNAR), but with all columns set to continuous data type.

In [ ]:
from mechanism_viewer import generate_dataset_with_missing_data, ColType

n_complete_cols = 1

data3 = generate_dataset_with_missing_data(100, [ColType.CONTINUOUS ,ColType.CONTINUOUS, ColType.CONTINUOUS, ColType.CONTINUOUS], n_complete_cols, ["MAR", "MCAR", "MNAR"], [0.2, 0.2, 0.2])
display(data3.head(10))

,Col1,Col2,Col3,Col4
0,0.496714,-1.415371,0.357787,-0.828995
1,-0.138264,-0.420645,0.560785,-0.560181
2,0.647689,-0.342715,1.083051,0.747294
3,1.523030,NaN,1.053802,0.610370
4,-0.234153,-0.161286,-1.377669,-0.020902
5,-0.234137,0.404051,-0.937825,0.117327
6,1.579213,NaN,NaN,NaN
7,0.767435,NaN,NaN,-0.591571
8,-0.469474,0.257550,NaN,0.547097
9,0.542560,-0.074446,3.852731,-0.202193
